In [0]:
from pyspark.sql.functions import *
import re

df = spark.table("bronze.procedures")


df = df.toDF(*[
    re.sub(r'[^a-z0-9_]', '', c.lower().replace(" ", "_"))
    for c in df.columns
])


# 3. HANDLE NULLS

df = df.replace(["", "unknown", "UNKNOWN"], None)

# 4. FIX BASE COST (REMOVE COMMAS)

df = df.withColumn(
    "base_cost",
    regexp_replace(col("base_cost"), ",", "")
)


# 5. FIX TIMESTAMPS 

df = df.withColumn("start", to_timestamp(col("start"))) \
       .withColumn("stop", to_timestamp(col("stop")))


# 6. CAST TYPES

df = df.withColumn("base_cost", col("base_cost").cast("double"))

df = df.withColumn("code", col("code").cast("double").cast("long")) \
       .withColumn("reasoncode", col("reasoncode").cast("double").cast("long"))

# 7. TRIM TEXT

df = df.withColumn("description", trim(col("description"))) \
       .withColumn("reasondescription", trim(col("reasondescription")))


# 8. REMOVE BAD RECORDS 

df = df.filter(
    col("patient").isNotNull() &
    col("encounter_id").isNotNull()
)

# 9. REMOVE NEGATIVE COSTS

df = df.filter(col("base_cost").isNull() | (col("base_cost") >= 0))

# 10. REMOVE SYSTEM COLUMNS

df = df.select([
    col(c) for c in df.columns if not c.startswith("_")
])


df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.procedures")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;